In [1]:
import os
from pathlib import Path
import random
import requests
from bs4 import BeautifulSoup

In [2]:
from itertools import islice

In [3]:
FOLDER = "policies"

In [4]:
policies = Path(FOLDER)
policies.mkdir(parents=True, exist_ok=True)

In [5]:
base_url = "https://licindia.in"
response = requests.get(f"{base_url}/web/guest/insurance-plan")

In [6]:
soup = BeautifulSoup(response.content, 'html.parser')

In [7]:
# random.seed(42)
# links = soup.find_all('a')
# policy_links = [link['href'] for link in links if 'LIC' in link.text and 'https' not in link['href'] and 'guest' in link['href']]
# policy_links[:] = policy_links[:5]
# policy_links

In [8]:
links = soup.find_all('a')
policy_links = {link.text.strip().replace("LIC's ", '').replace("LIC’s ", '').replace(' ', '_').replace('(', '') \
                .replace(')', '').replace(':', ''): link['href'] \
                for link in links if 'LIC' in link.text and 'https' not in link['href'] and 'guest' in link['href']}

#policy_links = {re.sub(r'\(\w+:\)', '', p): l for p, l in policy_links.items()}
#policy_links = dict(islice(policy_links.items(), 7))
policy_links

{'Single_Premium_Endowment_Plan': '/web/guest/lic-s-single-premium-endowment-plan-717-512n283v03',
 'New_Endowment_Plan': '/web/guest/lic-s-new-endowment-plan-714-512n277v03',
 'New_Jeevan_Anand': '/web/guest/lic-s-new-jeevan-anand-715%09512n279v03',
 'Jeevan_Lakshya': '/web/guest/lic-s-jeevan-lakshya-733-512n297v03',
 'Jeevan_Labh_Plan': '/web/guest/lic-s-jeevan-labh-plan-736-512n304v03',
 'Amritbaal': '/web/guest/lic-s-amritbaal-774%09512n365v02',
 'Bima_Jyoti': '/web/guest/lic-s-bima-jyoti-new',
 'Nav_Jeevan_Shree': '/web/guest/lic-s-nav-jeevan-shree-plan-no.912-1',
 'Bima_Lakshmi': '/web/guest/lic-s-bima-lakshmi-881-512n389v01',
 'Jeevan_Umang': '/web/guest/lic-sjeevanumang-745%09512n312v03',
 'Jeevan_Utsav': '/web/guest/lic-s-jeevan-utsav1',
 'Jeevan_Utsav_Single_premium': '/web/guest/lic-s-jeevan-utsav-single-premium',
 'Bima_Shree': '/web/guest/lic-s-bima-shree',
 'New_Money_Back_Plan-_20_Years': '/web/guest/lic-s-new-money-back-plan-20-years',
 'New_Money_Back_Plan-25_years': '

In [9]:
policy_pdfs = {}
for p, l in policy_links.items():
    url = base_url + l
    s = BeautifulSoup(requests.get(url).content, 'html.parser')
    pdf = [link['href'] for link in s.find_all('a') if "Policy Document" in link.text][0]
    pdf = pdf[: pdf.index('pdf') + 3]
    policy_pdfs[p] = pdf

In [10]:
policy_pdfs

{'Single_Premium_Endowment_Plan': '/documents/20121/1243952/Final+Policy+doc_LIC%27s+New+SP+Endowment_V03_website.pdf',
 'New_Endowment_Plan': '/documents/20121/1243952/Final+Policy+doc_LIC%27s+New+Endowment_V03_website.pdf',
 'New_Jeevan_Anand': '/documents/20121/1243952/Final+Policy+Doc_LIC%27s+New+Jeevan+Anand_V03_website.pdf',
 'Jeevan_Lakshya': '/documents/20121/1243952/Policy+Document_LIC%27s+Jeevan+Lakshya_CC.pdf',
 'Jeevan_Labh_Plan': '/documents/20121/1243952/Final_Policy+Docs_LIC%27s+Jeevan+Labh_V03_website.pdf',
 'Amritbaal': '/documents/20121/1243952/Final_+Policy+Document+LIC%27s+Amritbaal+300924_cc_with_logo.pdf',
 'Bima_Jyoti': '/documents/20121/1243952/Final+Policy+Document+LIC%27s+Bima+Jyoti+V03_website.pdf',
 'Nav_Jeevan_Shree': '/documents/20121/1385382/Policy+Document_Nav+Jeevan+Shree_512N387V02.pdf',
 'Bima_Lakshmi': '/documents/20121/1385382/Policy+Document_LIC%27s+Bima+Lakshmi.pdf',
 'Jeevan_Umang': '/documents/20121/1240315/Policy_doc_Jeevan_Umang_290924-withlog

In [ ]:
for p, d in policy_pdfs.items():
    with open(f"{policies}/{p}.pdf", 'wb') as f:
        url = base_url + d
        f.write(requests.get(url).content)